#### Note: In the notebook **application_sme**, when the **primary.Application** table is created, the **BuisnessPlansIDS** of the table **business_plan_rawdata** are filtered with inner join on the table **business_plan**. This step is done though in the stored procedure that creates the table: **business_plan_rawdata**. To validate the claim, check the analysis below:

In [0]:
%sql
WITH sme_business_plan_rawdata AS (
SELECT * FROM `p-cdh`.landed.sme_business_plan_rawdata WHERE BusinessDate  = '2026-02-11'),

sme_business_plan AS (
SELECT * FROM `p-cdh`.landed.sme_business_plan WHERE BusinessDate  = '2026-02-11')

-- Rows in the table sme_business_plan_rawdata: 8219
-- Rows after the inner join: 8219
-- Therefore, the INNER JOIN in the application_sme notebook is redundant
SELECT COUNT(*) AS countRows
FROM sme_business_plan_rawdata AS x
INNER JOIN sme_business_plan AS y
ON x.BusinessPlanID = y.ID

#### Tables checked:
##### 1. Business_Plan_Corp_Bankruptcy 
##### 2. Business_Plan_Indv_Bankcruptcy
##### 3. Business_Plan_Perimeter_LBS_Settlement_final
##### 4. Business_Plan_Stand_Still
##### 5. Business_Plan_Vol_CBS_NonRE
##### 6. Business_Plan_Vol_CBS_RE_3rdParty
##### 7. Business_Plan_Vol_CBS_REO_Sale
##### 8. Business_Plan_Vol_CBS_REO_Hold
##### 9. Business_Plan_Forc_CBS_REO_Sale
##### 10. Business_Plan_Forc_CBS_NonRE
##### 11. Business_Plan_Forc_CBS_REO_Hold
##### 12. Business_Plan_Forc_CBS_RE_3rdParty

#### 1. Investigate table: **Business_Plan_Corp_Bankruptcy** (Landing/Landed: **sme_business_plan_corp_bankruptcy**)

In [0]:
%sql
SELECT BusinessPlanID, -- Display explicitly all the columns in the table
        DebtorCDI,
        AgreementNumber,
        BusinessPlan_Status,
        BusinessPlanStrategy_Code,
        BusinessPlan_SolutionType,
        BusinessPlan_Comments,
        BusinessPlanStrategy_Status,
        Confidence,
        ProposedBy,
        BusinessPlanStrategy_StrategyType,
        BusinessPlanStrategy_Comments,
        LastModifiedDate,
        CollectionPlan_Month,
        CollectionPlan_Amount,
        PortfolioID,
        BusinessDate
 LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_corp_bankruptcy -- Count rows: 456
WHERE BusinessDate  = '2026-02-06' --'2026-02-11'
AND AgreementNumber = 'EUC1757008732'
AND BusinessPlanID = '290'
AND CollectionPlan_Month = '2035-11-01'
ORDER BY BusinessPlanStrategy_Code, CollectionPlan_Month DESC
LIMIT 10
-- 1. Columns of the table: spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_corp_bankruptcy """).columns
-- 2. Keys of the table: **BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, CollectionPlan_Month**
-- 3. However, how do we handle the following 1 special case?: 
/*
SELECT *
FROM  DWH_STG.sme.Business_Plan_Corp_Bankruptcy
WHERE AgreementNumber = 'EUC1757008732'
AND BusinessPlanID = '290'
AND CollectionPlan_Month = '2035-11-01'
ORDER BY BusinessPlanStrategy_Code, CollectionPlan_Month DESC
*/

In [0]:
%sql
with sme_business_plan_corp_bankruptcy AS (
SELECT BusinessPlanID,
DebtorCDI,
AgreementNumber,
BusinessPlan_Status,
BusinessPlanStrategy_Code, --BP_Strategy_Code
BusinessPlan_SolutionType,
BusinessPlan_Comments,
BusinessPlanStrategy_Status,
Confidence,
ProposedBy,
BusinessPlanStrategy_StrategyType,
BusinessPlanStrategy_Comments,
LastModifiedDate,
CollectionPlan_Month,
CollectionPlan_Amount,
PortfolioID,
BusinessDate,
LoadingDateTime,
ROW_NUMBER () OVER (PARTITION BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType order by BusinessPlanStrategy_Code desc) AS rnk
FROM `p-cdh`.landed.sme_business_plan_corp_bankruptcy
WHERE BusinessDate  = '2026-02-12'
)

-- Υπάρχουν μόνο 2 εγγραφές με αυτή την ιδιότητα:
-- 290	EUC1757008732	Application	Corp. Bankruptcy (art.99, art.106)	2
-- 567	NEP1825923191-0000000001466153	Cancelled	Corp. Bankruptcy (art.99, art.106)	2
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, COUNT(DISTINCT BusinessPlanStrategy_Code) AS DistinctBusinessPlanStrategy_Code
FROM sme_business_plan_corp_bankruptcy
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType
ORDER BY DistinctBusinessPlanStrategy_Code DESC
LIMIT 10
*/

-- Aς ελέγξουμε αυτές τις 2 εγγραφές:
-- ΠΡΟΣΟΧΗ: Δεν βγαζει νόημα το deduplication που έχει στο Stored Procedure στην DWH_STG
/*
SELECT CollectionPlan_Month, CollectionPlan_Amount, *
FROM sme_business_plan_corp_bankruptcy
WHERE BusinessPlanID = 290
AND AgreementNumber = 'EUC1757008732'
AND BusinessPlan_Status = 'Application'
AND BusinessPlanStrategy_StrategyType = 'Corp. Bankruptcy (art.99, art.106)'
ORDER BY BusinessPlanStrategy_Code DESC, CollectionPlan_Month DESC
*/


SELECT CollectionPlan_Month, CollectionPlan_Amount, *
FROM sme_business_plan_corp_bankruptcy
WHERE BusinessPlanID = 567
AND AgreementNumber = 'NEP1825923191-0000000001466153'
AND BusinessPlan_Status = 'Cancelled'
AND BusinessPlanStrategy_StrategyType = 'Corp. Bankruptcy (art.99, art.106)'
ORDER BY BusinessPlanStrategy_Code DESC, CollectionPlan_Month DESC


#### Question: Why the **rank** is conducted this way in the respective stored procedure concerning the table? - It does not seem right: **Business_Plan_Corp_Bankruptcy** (Landing/Landed: **sme_business_plan_corp_bankruptcy**) - Perhaps because in such a way we eliminate all the entries/rows with **CANCELLED BUSINESS PLAN STATUSES**. Let's check this assumption

#### Answer: According to the test below, this is not the case

In [0]:
%sql
SELECT DISTINCT BusinessPlan_Status
FROM (
SELECT *,
       ROW_NUMBER () OVER (PARTITION BY BusinessPlanID,
                                        AgreementNumber,
                                        BusinessPlan_Status,
                                        BusinessPlanStrategy_StrategyType
                          ORDER BY BusinessPlanStrategy_Code DESC) AS rnk
FROM `p-cdh`.landed.sme_business_plan_corp_bankruptcy
WHERE BusinessDate  = '2026-02-12'
)
WHERE rnk <> 1

#### **Question/Hypothesis**: why the RANK is conducted in the code this way - perhaps we take the **oldest or newest business plans** - that is Business Plans with oldest or newest Collection Plan Month

In [0]:
%sql
with sme_business_plan_corp_bankruptcy AS (
SELECT BusinessPlanID,
DebtorCDI,
AgreementNumber,
BusinessPlan_Status,
BusinessPlanStrategy_Code, --BP_Strategy_Code
BusinessPlan_SolutionType,
BusinessPlan_Comments,
BusinessPlanStrategy_Status,
Confidence,
ProposedBy,
BusinessPlanStrategy_StrategyType,
BusinessPlanStrategy_Comments,
LastModifiedDate,
CollectionPlan_Month,
CollectionPlan_Amount,
PortfolioID,
BusinessDate,
LoadingDateTime,
ROW_NUMBER () OVER (PARTITION BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType order by BusinessPlanStrategy_Code desc) AS rnk
FROM `p-cdh`.landed.sme_business_plan_corp_bankruptcy
WHERE BusinessDate  = '2026-02-12'
),

-- ################################################################
-- ############################ Check 1 ###########################
-- ################################################################
/*
sme_business_plan_corp_bankruptcy_with_rnk_equal_one AS (
SELECT BusinessPlanID, AgreementNumber, DebtorCDI,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType, CollectionPlan_Month
FROM sme_business_plan_corp_bankruptcy
WHERE rnk = 1
),

sme_business_plan_corp_bankruptcy_with_rnk_unequal_one AS (
SELECT BusinessPlanID,
       AgreementNumber,
       DebtorCDI,
       BusinessPlan_Status,
       BusinessPlanStrategy_StrategyType,
       MIN(CollectionPlan_Month) AS Min_CollectionPlan_Month,
       MAX(CollectionPlan_Month) AS Max_CollectionPlan_Month
FROM sme_business_plan_corp_bankruptcy
WHERE rnk <> 1 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType
)
*/

/*
SELECT x.*, y.Min_CollectionPlan_Month
FROM sme_business_plan_corp_bankruptcy_with_rnk_equal_one AS x
LEFT JOIN sme_business_plan_corp_bankruptcy_with_rnk_unequal_one AS y
ON x.BusinessPlanID = y.BusinessPlanID
AND x.AgreementNumber = y.AgreementNumber
AND x.DebtorCDI = y.DebtorCDI
AND x.BusinessPlan_Status = y.BusinessPlan_Status
AND x.BusinessPlanStrategy_StrategyType = y.BusinessPlanStrategy_StrategyType
*/

/*
SELECT *
FROM sme_business_plan_corp_bankruptcy
WHERE AgreementNumber = 'NEP1825923191-0000000001466153'
ORDER BY BusinessPlanStrategy_Code DESC, CollectionPlan_Month DESC
*/


-- ################################################################
-- ############################ Check 2 ###########################
-- ################################################################

intermediary_sme_business_plan_corp_bankruptcy AS (
SELECT BusinessPlanID,
       AgreementNumber,
       DebtorCDI,
       BusinessPlan_Status,
       BusinessPlanStrategy_StrategyType,
       BusinessPlanStrategy_Code,
       COUNT(DISTINCT CollectionPlan_Month) AS countDistinctCollectionPlan_Month,
       MIN(CollectionPlan_Month) AS Min_CollectionPlan_Month, 
       MAX(CollectionPlan_Month) AS Max_CollectionPlan_Month,
       ROW_NUMBER() OVER(PARTITION BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType ORDER BY BusinessPlanStrategy_Code DESC) AS rnk
FROM sme_business_plan_corp_bankruptcy
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
),

intermediary_sme_business_plan_corp_bankruptcy_with_rnk_equal_one AS (
SELECT BusinessPlanID, AgreementNumber, DebtorCDI,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, Min_CollectionPlan_Month
FROM intermediary_sme_business_plan_corp_bankruptcy
WHERE rnk = 1
),

intermediary_sme_business_plan_corp_bankruptcy_with_rnk_unequal_one AS (
SELECT BusinessPlanID, AgreementNumber, DebtorCDI,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, Min_CollectionPlan_Month
FROM intermediary_sme_business_plan_corp_bankruptcy
WHERE rnk <> 1
)

SELECT *
FROM (
SELECT x.BusinessPlanID, 
       x.AgreementNumber,
       x.DebtorCDI,
       x.BusinessPlan_Status,
       x.BusinessPlanStrategy_StrategyType,
       x.BusinessPlanStrategy_Code, 
       y.BusinessPlanStrategy_Code AS SecondaryBusinessPlanStrategy_Code, 
       x.Min_CollectionPlan_Month,
       y.Min_CollectionPlan_Month AS SecondaryMin_CollectionPlan_Month
FROM intermediary_sme_business_plan_corp_bankruptcy_with_rnk_equal_one AS x
INNER JOIN intermediary_sme_business_plan_corp_bankruptcy_with_rnk_unequal_one AS y
ON x.BusinessPlanID = y.BusinessPlanID
AND x.AgreementNumber = y.AgreementNumber
AND x.DebtorCDI = y.DebtorCDI
AND x.BusinessPlan_Status = y.BusinessPlan_Status
AND x.BusinessPlanStrategy_StrategyType = y.BusinessPlanStrategy_StrategyType
) AS t
WHERE BusinessPlanStrategy_Code <> SecondaryBusinessPlanStrategy_Code
AND Min_CollectionPlan_Month < SecondaryMin_CollectionPlan_Month -- IMPORTANT: **If it does not return empty table than the assumption does not stand**


/*
SELECT BusinessPlanID, 
       AgreementNumber,
       BusinessPlan_Status,
       BusinessPlanStrategy_StrategyType,
       COUNT(DISTINCT BusinessPlanStrategy_Code) AS countDistinctBusinessPlanStrategy_Codes
FROM sme_business_plan_corp_bankruptcy
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType
ORDER BY countDistinctBusinessPlanStrategy_Codes DESC
LIMIT 5
*/

/*
SELECT *
FROM intermediary_sme_business_plan_corp_bankruptcy
WHERE AgreementNumber = 'EUC1757008732' -- Alternative AgreementNumber = 'NEP1825923191-0000000001466153'
ORDER BY BusinessPlanStrategy_Code DESC
*/

In [0]:
%sql
SELECT *
FROM `p-cdh`.landed.sme_business_plan_corp_bankruptcy
WHERE BusinessDate  = '2026-02-12'
AND AgreementNumber = 'SYM9677010176' 
AND DebtorCDI = 'SYM733216161'  
ORDER BY BusinessPlanStrategy_Code DESC, CollectionPlan_Month DESC

#### Question: Try to justify the inner join in the application_sme notebook between **`p-cdh`.landed.sme_business_plan** and **`p-cdh`.landed.sme_business_plan_rawdata**

In [0]:
%sql
SELECT DISTINCT Business_plan_status
FROM `p-cdh`.landed.sme_business_plan
WHERE BusinessDate  = '2026-02-12'

2. Investigate table: **Business_Plan_Forc_CBS_RE_3rdParty** (Landing/Landed: **sme_business_plan_forc_cbs_re_3rdparty**)

In [0]:
%sql
SELECT BusinessPlanID,
        DebtorCDI,
        AgreementNumber,
        BusinessPlan_Status,
        BusinessPlanStrategy_Code,
        BusinessPlan_SolutionType,
        BusinessPlan_Comments,
        BusinessPlanStrategy_Status,
        Confidence,
        ProposedBy,
        BusinessPlanStrategy_StrategyType,
        BusinessPlanStrategy_Comments,
        LastModifiedDate,
        PortfolioID,
        TimeToAuction,
        AuctionPrice,
        NetRecoveryAmount,
        BusinessDate,
        LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_re_3rdparty 
WHERE BusinessDate  = '2026-02-12' 
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_re_3rdparty WHERE BusinessDate  = '2026-02-12' """).count() # Count rows: 5570
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_re_3rdparty WHERE BusinessDate  = '2026-02-12' """).columns
-- 1. Keys of the table: BusinessPlanID, AgreementNumber, (DebtorCDI ?), BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code (To be discussed)
-- 2. No special cases for the selection of the keys, with exception the inclusion or not of the column: DebtorCDI
/*
SELECT TOP 10 BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM  DWH_STG.sme.Business_Plan_Forc_CBS_RE_3rdParty 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

In [0]:
%sql
/*
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_re_3rdparty 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/

-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_re_3rdparty 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 3. Investigate table: **Business_Plan_Forc_CBS_REO_Sale** (Landing/Landed: **sme_business_plan_forc_cbs_reo_sale**)

In [0]:
%sql
SELECT BusinessPlanID,
        DebtorCDI,
        AgreementNumber,
        BusinessPlan_Status,
        BusinessPlanStrategy_Code,
        BusinessPlan_SolutionType,
        BusinessPlan_Comments,
        BusinessPlanStrategy_Status,
        Confidence,
        ProposedBy,
        BusinessPlanStrategy_StrategyType,
        BusinessPlanStrategy_Comments,
        LastModifiedDate,
        PortfolioID,
        TimeToAuction,
        AuctionPrice,
        TimeToSell,
        NetRecoveryAmount,
        BusinessDate,
        LoadingDateTime
FROM  `p-cdh`.landed.sme_business_plan_forc_cbs_reo_sale 
WHERE BusinessDate  = '2026-02-12'
LIMIT 10
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_sale WHERE BusinessDate  = '2026-02-12' """).count() # Number of rows: 583
--spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_sale WHERE BusinessDate  = '2026-02-12' """).columns


In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_sale 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/


-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_sale 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 4.Investigate table: **Business_Plan_Forc_CBS_NonRE** (Landing/Landed: **sme_business_plan_forc_cbs_nonre**)

In [0]:
%sql
SELECT BusinessPlanID,
      DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      PortfolioID,
      TimeToRecovery,
      NetRecoveryAmount,
      BusinessDate,
      LoadingDateTime
FROM  `p-cdh`.landed.sme_business_plan_forc_cbs_nonre 
WHERE BusinessDate  = '2026-02-12'
LIMIT 10
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_nonre WHERE BusinessDate  = '2026-02-12' """).count() # Number of rows: 76
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_nonre WHERE BusinessDate  = '2026-02-12' """).columns

/*
-- Potential Keys (to be discussed): BusinessPlanID, AgreementNumber, (DebtorCDI ?), BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
SELECT TOP 10 BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM  DWH_STG.sme.Business_Plan_Forc_CBS_NonRE
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_nonre 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/


-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_nonre 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 5.Investigate table: **Business_Plan_Forc_CBS_REO_Hold** (Landing/Landed: **sme_business_plan_forc_cbs_reo_hold**)

In [0]:
%sql
SELECT BusinessPlanID,
      DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      PortfolioID,
      TimeToAuction,
      AuctionPrice,
      TimeToSell,
      NetRecoveryAmount,
      BusinessDate,
      LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_hold 
WHERE BusinessDate  = '2026-02-12'
LIMIT 10
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_hold WHERE BusinessDate  = '2026-02-12' """).count() # Number of rows: 12
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_hold WHERE BusinessDate  = '2026-02-12' """).columns

/*
-- Potential keys in the table: BusinessPlanID, AgreementNumber, (DebtorCDI ?), BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
SELECT BusinessPlanID,
       AgreementNumber,
       DebtorCDI,
       BusinessPlan_Status,
       BusinessPlanStrategy_StrategyType,
       BusinessPlanStrategy_Code,
       COUNT(*) AS countRows
FROM DWH_STG.sme.Business_Plan_Forc_CBS_REO_Hold
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/


In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_hold 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/

-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_forc_cbs_reo_hold 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 6.Investigate table: **Business_Plan_Vol_CBS_REO_Hold** (Landing/Landed: **sme_business_plan_vol_cbs_reo_hold**)
#### **Important note:** The table is empty, as well as the respective ITStaging table: **ITStaging.cepalplatform.SmeVBusinessPlanDataVolCbsReoHold**

In [0]:
%sql
SELECT BusinessPlanID,
      DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      PortfolioID,
      TimeToSell,
      NetRecoveryAmount,
      BusinessDate,
      LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_reo_hold 
--WHERE BusinessDate  = '2026-02-12' 
LIMIT 10

-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_vol_cbs_reo_hold WHERE BusinessDate  = '2026-02-12' """).count() --Number of rows: 12
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_vol_cbs_reo_hold WHERE BusinessDate  = '2026-02-12' """).columns

#### 7.Investigate table: **Business_Plan_Vol_CBS_REO_Sale** (Landing/Landed: **sme_business_plan_vol_cbs_reo_sale**) 

In [0]:
%sql
SELECT BusinessPlanID,
      DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      TimeToSell,
      NetRecoveryAmount,
      PortfolioID,
      BusinessDate,
      LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_reo_sale 
WHERE BusinessDate  = '2026-02-12' 
LIMIT 10
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_vol_cbs_reo_sale WHERE BusinessDate  = '2026-02-12' """).columns

In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_reo_sale 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/

-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_reo_sale 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 8.Investigate table: Business_Plan_Vol_CBS_NonRE (Landing/Landed: sme_business_plan_vol_cbs_nonre)

In [0]:
%sql
SELECT 
BusinessPlanID,
DebtorCDI,
AgreementNumber,
BusinessPlan_Status,
BusinessPlanStrategy_Code,
BusinessPlan_SolutionType,
BusinessPlan_Comments,
BusinessPlanStrategy_Status,
Confidence,
ProposedBy,
BusinessPlanStrategy_StrategyType,
BusinessPlanStrategy_Comments,
LastModifiedDate,
PortfolioID,
TimeToRecovery,
NetRecoveryAmount,
BusinessDate,
LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_nonre 
WHERE BusinessDate  = '2026-02-12'
LIMIT 2
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_vol_cbs_nonre WHERE BusinessDate  = '2026-02-12' """).columns

In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_nonre 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/

-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_nonre 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 9.Investigate table: Business_Plan_Vol_CBS_RE_3rdParty (Landing/Landed: sme_business_plan_vol_cbs_re_3rdparty)

In [0]:
%sql
SELECT 
BusinessPlanID,
DebtorCDI,
AgreementNumber,
BusinessPlan_Status,
BusinessPlanStrategy_Code,
BusinessPlan_SolutionType,
BusinessPlan_Comments,
BusinessPlanStrategy_Status,
Confidence,
ProposedBy,
BusinessPlanStrategy_StrategyType,
BusinessPlanStrategy_Comments,
LastModifiedDate,
PortfolioID,
TimeToSell,
NetRecoveryAmount,
BusinessDate,
LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_re_3rdparty 
WHERE BusinessDate  = '2026-02-12'
LIMIT 10
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_vol_cbs_re_3rdparty WHERE BusinessDate  = '2026-02-12' """).columns

In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_re_3rdparty 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/

-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_vol_cbs_re_3rdparty 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 10.Investigate table: **Business_Plan_Indv_Bankcruptcy** (Landing/Landed: **sme_business_plan_indv_bankcruptcy**)

In [0]:
%sql
SELECT BusinessPlanID,
      DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      PortfolioID,
      NumberOfInstallments,
      Installment,
      PrepaymentMonth,
      PrepaymentAmount,
      PeriodicityID,
      BusinessDate,
      LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_indv_bankcruptcy 
WHERE BusinessDate  = '2026-02-12'
LIMIT 10
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_indv_bankcruptcy WHERE BusinessDate  = '2026-02-12'""").columns

In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_indv_bankcruptcy 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/

-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_indv_bankcruptcy 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 11.Investigate table: **Business_Plan_Stand_Still** (Landing/Landed: **sme_business_plan_stand_still**)

In [0]:
%sql
SELECT BusinessPlanID,
      DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      PortfolioID,
      BusinessDate,
      LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_stand_still 
WHERE BusinessDate  = '2026-02-12'
LIMIT 10
-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_stand_still WHERE BusinessDate  = '2026-02-12'""").columns

In [0]:
%sql
-- Check 1: Ποιά είναι τα κλειδιά του πίνακα?
-- Απάντηση: είναι οι παρακάτω κολώνες: BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_indv_bankcruptcy 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code
ORDER BY countRows DESC
LIMIT 10
*/

-- Check 2: 
-- Νote: Το παρακάτω query επιστρέφει countRows = 1, για όλα τα rows
/*
SELECT BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_indv_bankcruptcy 
WHERE BusinessDate  = '2026-02-12' 
GROUP BY BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code
ORDER BY countRows DESC
*/

#### 12.Investigate table: **Business_Plan_Perimeter_LBS_Settlement_final** (Landing/Landed: **sme_business_plan_perimeter_lbs_settlement_final**)

In [0]:
%sql
WITH sme_business_plan_perimeter_lbs_settlement_final AS (
SELECT id, 
      LBS_ID,
      xLBS_ID,
      LBSParentID,
      DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      InstallmentDate,
      CollectionPlan_Month,
      CollectionPlan_Amount,
      PortfolioID,
      BusinessPlanStrategy_SolutionType,
      NumberOfInstallments,
      Installment,
      PrepaymentMonth,
      PrepaymentAmount,
      BaseInterestRate,
      BaseInterestRateValue,
      Spread,
      N128Percentage,
      TotalRateWithoutN128,
      Interest,
      PeriodicityID,
      InterestPaymentPeriodicityID,
      BalloonAmount,
      FirstInstallmentDate,
      InstallmentSN,
      SubProduct,
      insertDate,
      StartingCapitalAmn,
      BusinessDate,
      LoadingDateTime
FROM `p-cdh`.landed.sme_business_plan_perimeter_lbs_settlement_final 
WHERE BusinessDate  = '2026-02-12'
)

SELECT *
FROM sme_business_plan_perimeter_lbs_settlement_final
LIMIT 10
--spark.sql("""SELECT * FROM `p-cdh`.landed.sme_business_plan_perimeter_lbs_settlement_final WHERE BusinessDate  = '2026-02-12'""").columns
-- NOTES:
-- 1. The Column xLBS_ID is a duplicate of the column: LBS_ID, therefore it will be removed from the table: sme_business_plan_perimeter_lbs_settlement_final

In [0]:
%sql
-- Check 1:
/*
SELECT LBSParentID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, CollectionPlan_Month, COUNT(*) AS countRows
FROM `p-cdh`.landed.sme_business_plan_perimeter_lbs_settlement_final 
WHERE BusinessDate  = '2026-02-12' 
AND CollectionPlan_Month IS NOT NULL
GROUP BY LBSParentID, AgreementNumber, DebtorCDI, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, CollectionPlan_Month
ORDER BY countRows DESC
LIMIT 10
*/

-- The query returns two rows with 2 distinct IDS = 3518, 5048
-- THE FOLLOWING TWO RESULTS DOES NOT ALIGHN
/*
SELECT *
FROM `p-cdh`.landed.sme_business_plan_perimeter_lbs_settlement_final 
WHERE BusinessDate  = '2026-02-12' 
AND AgreementNumber = 'SYM2057014554'
AND DebtorCDI = 'SYM7637045048'
*/

/*
SELECT *
FROM `p-cdh`.landed.sme_business_plan_rawdata 
WHERE BusinessDate  = '2026-02-12' 
AND tbl = "Business_Plan_Perimeter_LBS_Settlement_final"
AND AgreementNumber = 'SYM2057014554'
AND DebtorCDI = 'SYM7637045048'
*/

#### 13.Investigate table: **BusinessPlanData_LBS_PP** (Landing/Landed: **sme_businessplandata_lbs_pp**)

In [0]:
%sql
WITH sme_businessplandata_lbs_pp AS (
SELECT DebtorCDI,
      AgreementNumber,
      BusinessPlan_Status,
      BusinessPlanStrategy_Code,
      BusinessPlan_CreationDate,
      BusinessPlan_SolutionType,
      BusinessPlan_Comments,
      BusinessPlanStrategy_Status,
      Confidence,
      ProposedBy,
      BusinessPlanStrategy_StrategyType,
      BusinessPlanStrategy_Comments,
      LastModifiedDate,
      BusinessPlanStrategy_Solution_Type,
      InstallmentTotalAmount,
      InstallmentDate,
      PortfolioID,
      BusinessDate,
      LoadingDateTime
FROM `p-cdh`.landed.sme_businessplandata_lbs_pp 
WHERE BusinessDate  = '2026-02-12'
)


SELECT  AgreementNumber,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, InstallmentDate, COUNT(*) AS countRows
FROM sme_businessplandata_lbs_pp
GROUP BY AgreementNumber,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, InstallmentDate
ORDER BY countRows DESC
LIMIT 5

-- spark.sql("""SELECT * FROM `p-cdh`.landed.sme_businessplandata_lbs_pp WHERE BusinessDate  = '2026-02-12'""").columns
-- Possible keys of the table: AgreementNumber, (DebtorCDI ?), BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, InstallmentDate with sum on InstallmentTotalAmount

/*
-- The following query has per: AgreementNumber, DebtorCDI,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code multiple InstallmentDates & InstallmentTotalAmounts
SELECT AgreementNumber, DebtorCDI,  BusinessPlan_Status, BusinessPlanStrategy_StrategyType,  BusinessPlanStrategy_Code, *
FROM sme_businessplandata_lbs_pp
WHERE AgreementNumber = 'NEP2006843691-0000000000014273'
AND DebtorCDI = 'NEP0015690959'
AND BusinessPlan_Status = 'Cancelled'
AND BusinessPlanStrategy_StrategyType = 'LBS - DPO'
AND BusinessPlanStrategy_Code = '1030'
ORDER BY InstallmentDate DESC
*/


/*
-- QUESTION: Πως χειριζόμαστε αυτές τις εγγραφές? - προσθέτουμε τα InstallmentTotalAmount ανά InstallmentDate?
SELECT  *
FROM sme_businessplandata_lbs_pp
WHERE AgreementNumber = 'NEP7190565291-0000002600450720'
AND DebtorCDI = 'NEP0010961783'
AND BusinessPlan_Status = 'Intention'
AND BusinessPlanStrategy_StrategyType = 'LBS - Ρύθμιση'
AND BusinessPlanStrategy_Code = '6202'
AND InstallmentDate = '2028-12-31'
*/

#### Create the stored procedure

In [0]:
import os, sys
import logging
import datetime
from datetime import datetime
from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession, DataFrame
from typing import Optional
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    expr,
    from_utc_timestamp,
    lit,
    max,
    sum,
    col,
    sha2,
    substring,
    to_date,
    when,
    broadcast,
    trim,
    row_number
)

def load_landed_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    # Trim all string columns
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

In [0]:
cluster_columns = ["BusinessDate"]
run_date = '2026-02-06'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = "`d-cdh`.landed"

# ============================== cepalplatform_smevbusinessplandatacorpbankruptsy ====================================

business_plan_corp_bankruptcy_extra_filters = None
if business_plan_corp_bankruptcy_extra_filters is not None:
    business_plan_corp_bankruptcy_extra_filters.append(date_filter)
    business_plan_corp_bankruptcy_final_filters = " AND ".join(
        business_plan_corp_bankruptcy_extra_filters
    )
else:
    business_plan_corp_bankruptcy_final_filters = date_filter

business_plan_corp_bankruptcy_selected_filters = [
    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "CollectionPlan_Month",
    "CollectionPlan_Amount",
    "PortfolioID",
    "BusinessDate",
    ]

# Keys of the table: **BusinessPlanID, AgreementNumber, BusinessPlan_Status, BusinessPlanStrategy_StrategyType, BusinessPlanStrategy_Code, CollectionPlan_Month**
business_plan_corp_bankruptcy = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandatacorpbankruptsy",
    business_plan_corp_bankruptcy_final_filters,
    business_plan_corp_bankruptcy_selected_filters,
    )

agg_business_plan_corp_bankruptcy = (
    business_plan_corp_bankruptcy.groupBy(
        "BusinessPlan_ID",
        "AgreementNumber",
        "BusinessPlan_Status",
        "BusinessPlanStrategy_StrategyType",
        "BusinessPlanStrategy_Code",
        "CollectionPlan_Month"
    )
    .agg(sum("CollectionPlan_Amount").alias("CollectionPlan_Total_Amount"))
)

window_spec = Window.partitionBy(
    "BusinessPlan_ID",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Code",
    "CollectionPlan_Month"
    ).orderBy("CollectionPlan_Amount")

dedup_business_plan_corp_bankruptcy = (
    business_plan_corp_bankruptcy.withColumn("rank", row_number().over(window_spec)).filter(col("rank") == 1)
    )

business_plan_corp_bankruptcy_join_condition = (
    (col("dedup.BusinessPlan_ID") == col("agg.BusinessPlan_ID"))
        &(col("dedup.AgreementNumber") == col("agg.AgreementNumber"))
        &(col("dedup.BusinessPlan_Status") == col("agg.BusinessPlan_Status"))
        &(col("dedup.BusinessPlanStrategy_StrategyType") == col("agg.BusinessPlanStrategy_StrategyType"))
        &(col("dedup.BusinessPlanStrategy_Code") == col("agg.BusinessPlanStrategy_Code"))
        &(col("dedup.CollectionPlan_Month") == col("agg.CollectionPlan_Month"))
)

business_plan_corp_bankruptcy_df = (
    dedup_business_plan_corp_bankruptcy.alias('dedup').join(
        agg_business_plan_corp_bankruptcy.alias("agg"),
        business_plan_corp_bankruptcy_join_condition,
        how="left"
    ).select(
        "dedup.*",
        "agg.CollectionPlan_Total_Amount"
         )
)


run_date = '2026-02-05' 
date_filter = f"businessdate = {run_date!r}"

# ============================== cepalplatform_smevbusinessplandataforccbsnonre ====================================

nonre_forc_extra_filters = None
if nonre_forc_extra_filters is not None:
    nonre_forc_extra_filters.append(date_filter)
    nonre_forc_final_filters = " AND ".join(
        nonre_forc_extra_filters
    )
else:
    nonre_forc_final_filters = date_filter

nonre_forc_selected_filters = [
    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "TimeToRecovery",
    "NetRecoveryAmount",
    "BusinessDate"
]

nonre_forc = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandataforccbsnonre",
    nonre_forc_final_filters,
    nonre_forc_selected_filters,
    )

# ==================================== cepalplatform_smevbusinessplandataforccbsre3rdparty =======================================

re_3rdparty_forc_extra_filters = None
if re_3rdparty_forc_extra_filters is not None:
    re_3rdparty_forc_extra_filters.append(date_filter)
    re_3rdparty_forc_final_filters = " AND ".join(
        re_3rdparty_forc_extra_filters
    )
else:
    re_3rdparty_forc_final_filters = date_filter

re_3rdparty_forc_selected_filters = [

    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "AuctionPrice",
    "TimeToAuction",
    "NetRecoveryAmount",
    "BusinessDate"
]

re_3rdparty_forc = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandataforccbsre3rdparty",
    re_3rdparty_forc_final_filters,
    re_3rdparty_forc_selected_filters,
    )

# ==================================== cepalplatform_smevbusinessplandataforccbsreohold =======================================

reo_hold_forc_extra_filters = None
if reo_hold_forc_extra_filters is not None:
    reo_hold_forc_extra_filters.append(date_filter)
    reo_hold_forc_final_filters = " AND ".join(
        reo_hold_forc_extra_filters
    )
else:
    reo_hold_forc_final_filters = date_filter

reo_hold_forc_selected_filters = [

    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "AuctionPrice",
    "TimeToAuction",
    "TimeToSell",
    "NetRecoveryAmount",
    "BusinessDate"
]

reo_hold_forc = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandataforccbsreohold",
    reo_hold_forc_final_filters,
    reo_hold_forc_selected_filters,
    )

# ==================================== cepalplatform_smevbusinessplandataforccbsreosale =======================================

reo_sale_forc_extra_filters = None
if reo_sale_forc_extra_filters is not None:
    reo_sale_forc_extra_filters.append(date_filter)
    reo_sale_forc_final_filters = " AND ".join(
        reo_sale_forc_extra_filters
    )
else:
    reo_sale_forc_final_filters = date_filter

reo_sale_forc_selected_filters = [

    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "AuctionPrice",
    "TimeToAuction",
    "TimeToSell",
    "NetRecoveryAmount",
    "BusinessDate"
]

reo_sale_forc = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandataforccbsreosale",
    reo_sale_forc_final_filters,
    reo_sale_forc_selected_filters,
    )

# ================== target_business_plan_corp_bankruptcy table schema ==================

target_business_plan_corp_bankruptcy = business_plan_corp_bankruptcy_df.select(
    business_plan_corp_bankruptcy_df.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    business_plan_corp_bankruptcy_df.DebtorCDI.cast("String"),
    business_plan_corp_bankruptcy_df.AgreementNumber.cast("String"),
    business_plan_corp_bankruptcy_df.BusinessPlan_Status.cast("String"),
    business_plan_corp_bankruptcy_df.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    business_plan_corp_bankruptcy_df.BusinessPlan_SolutionType.cast("String"),
    business_plan_corp_bankruptcy_df.BusinessPlan_Comments.cast("String"),
    business_plan_corp_bankruptcy_df.BusinessPlanStrategy_Status.cast("String"),
    business_plan_corp_bankruptcy_df.Confidence.cast("String"),
    business_plan_corp_bankruptcy_df.ProposedBy.cast("String"),
    business_plan_corp_bankruptcy_df.BusinessPlanStrategy_StrategyType.cast("String"),
    business_plan_corp_bankruptcy_df.BusinessPlanStrategy_Comments.cast("String"),
    business_plan_corp_bankruptcy_df.LastModifiedDate.cast("Date"),
    business_plan_corp_bankruptcy_df.CollectionPlan_Month.cast("Date"),
    business_plan_corp_bankruptcy_df.CollectionPlan_Total_Amount.alias("CollectionPlan_Amount").cast("Decimal(19, 2)"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    lit(None).cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    business_plan_corp_bankruptcy_df.PortfolioID.cast("Int"),
    business_plan_corp_bankruptcy_df.BusinessDate.cast("Date"),
    lit("Business_Plan_Corp_Bankruptcy").cast("String").alias("tbl"),
    )

# ================== nonre_forc table schema ==================

target_nonre_forc = nonre_forc.select(
    nonre_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    nonre_forc.DebtorCDI.cast("String"),
    nonre_forc.AgreementNumber.cast("String"),
    nonre_forc.BusinessPlan_Status.cast("String"),
    nonre_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    nonre_forc.BusinessPlan_SolutionType.cast("String"),
    nonre_forc.BusinessPlan_Comments.cast("String"),
    nonre_forc.BusinessPlanStrategy_Status.cast("String"),
    nonre_forc.Confidence.cast("String"),
    nonre_forc.ProposedBy.cast("String"),
    nonre_forc.BusinessPlanStrategy_StrategyType.cast("String"),
    nonre_forc.BusinessPlanStrategy_Comments.cast("String"),
    nonre_forc.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    nonre_forc.TimeToRecovery.cast("Int").alias("TimeToRecovery"),
    nonre_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    nonre_forc.PortfolioID.cast("Int"),
    nonre_forc.BusinessDate.cast("Date"),
    lit("Business_Plan_Forc_CBS_NonRE").cast("String").alias("tbl"),
)

# ================== re_3rdparty_forc table schema ==================

target_re_3rdparty_forc = re_3rdparty_forc.select(
    re_3rdparty_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlan_ID"),
    re_3rdparty_forc.DebtorCDI.cast("String"),
    re_3rdparty_forc.AgreementNumber.cast("String"),
    re_3rdparty_forc.BusinessPlan_Status.cast("String"),
    re_3rdparty_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    re_3rdparty_forc.BusinessPlan_SolutionType.cast("String"),
    re_3rdparty_forc.BusinessPlan_Comments.cast("String"),
    re_3rdparty_forc.BusinessPlanStrategy_Status.cast("String"),
    re_3rdparty_forc.Confidence.cast("String"),
    re_3rdparty_forc.ProposedBy.cast("String"),
    re_3rdparty_forc.BusinessPlanStrategy_StrategyType.cast("String"),
    re_3rdparty_forc.BusinessPlanStrategy_Comments.cast("String"),
    re_3rdparty_forc.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    re_3rdparty_forc.TimeToAuction.cast("Int").alias("TimeToAuction"),
    re_3rdparty_forc.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    re_3rdparty_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    re_3rdparty_forc.PortfolioID.cast("Int"),
    re_3rdparty_forc.BusinessDate.cast("Date"),
    lit("Business_Plan_Forc_CBS_RE_3rdParty").cast("String").alias("tbl"),
    )

# ================== reo_hold_forc table schema ==================

target_reo_hold_forc = reo_hold_forc.select(
    reo_hold_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlan_ID"),
    reo_hold_forc.DebtorCDI.cast("String"),
    reo_hold_forc.AgreementNumber.cast("String"),
    reo_hold_forc.BusinessPlan_Status.cast("String"),
    reo_hold_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    reo_hold_forc.BusinessPlan_SolutionType.cast("String"),
    reo_hold_forc.BusinessPlan_Comments.cast("String"),
    reo_hold_forc.BusinessPlanStrategy_Status.cast("String"),
    reo_hold_forc.Confidence.cast("String"),
    reo_hold_forc.ProposedBy.cast("String"),
    reo_hold_forc.BusinessPlanStrategy_StrategyType.cast("String"),
    reo_hold_forc.BusinessPlanStrategy_Comments.cast("String"),
    reo_hold_forc.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    reo_hold_forc.TimeToAuction.cast("Int").alias("TimeToAuction"),
    reo_hold_forc.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
    reo_hold_forc.TimeToSell.cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    reo_hold_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    reo_hold_forc.PortfolioID.cast("Int"),
    reo_hold_forc.BusinessDate.cast("Date"),
    lit("Business_Plan_Forc_CBS_REO_Hold").cast("String").alias("tbl"),
)

# ================== reo_sale_forc table schema ==================

target_reo_sale_forc = reo_sale_forc.select(
    reo_sale_forc.BusinessPlan_ID.cast("Int").alias("BusinessPlan_ID"),
    reo_sale_forc.DebtorCDI.cast("String"),
    reo_sale_forc.AgreementNumber.cast("String"),
    reo_sale_forc.BusinessPlan_Status.cast("String"),
    reo_sale_forc.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    reo_sale_forc.BusinessPlan_SolutionType.cast("String"),
    reo_sale_forc.BusinessPlan_Comments.cast("String"),
    reo_sale_forc.BusinessPlanStrategy_Status.cast("String"),
    reo_sale_forc.Confidence.cast("String"),
    reo_sale_forc.ProposedBy.cast("String"),
    reo_sale_forc.BusinessPlanStrategy_StrategyType.cast("String"),
    reo_sale_forc.BusinessPlanStrategy_Comments.cast("String"),
    reo_sale_forc.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    reo_sale_forc.TimeToAuction.cast("Int").alias("TimeToAuction"),
    reo_sale_forc.AuctionPrice.cast("Decimal(19,2)").alias("AuctionPrice"),
    reo_sale_forc.TimeToSell.cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    reo_sale_forc.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    reo_sale_forc.PortfolioID.cast("Int"),
    reo_sale_forc.BusinessDate.cast("Date"),
    lit("Business_Plan_Forc_CBS_REO_Sale").cast("String").alias("tbl"),
)

In [0]:

# ============================== cepalplatform_smevbusinessplandatavolcbsnonre ====================================

nonre_vol_extra_filters = None
if nonre_vol_extra_filters is not None:
    nonre_vol_extra_filters.append(date_filter)
    nonre_vol_final_filters = " AND ".join(
        nonre_vol_extra_filters
    )
else:
    nonre_vol_final_filters = date_filter

nonre_vol_selected_filters = [

    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "TimeToRecovery",
    "NetRecoveryAmount",
    "BusinessDate"
]

nonre_vol = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandatavolcbsnonre",
    nonre_vol_final_filters,
    nonre_vol_selected_filters,
    )

# ==================================== cepalplatform_smevbusinessplandatavolcbsre3rdparty =======================================

re_3rdparty_vol_extra_filters = None
if re_3rdparty_vol_extra_filters is not None:
    re_3rdparty_vol_extra_filters.append(date_filter)
    re_3rdparty_vol_final_filters = " AND ".join(
        re_3rdparty_vol_extra_filters
    )
else:
    re_3rdparty_vol_final_filters = date_filter

re_3rdparty_vol_selected_filters = [

    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "TimeToSell",
    "NetRecoveryAmount",
    "BusinessDate"
]

re_3rdparty_vol = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandatavolcbsre3rdparty",
    re_3rdparty_vol_final_filters,
    re_3rdparty_vol_selected_filters,
    )

# ==================================== cepalplatform_smevbusinessplandatavolcbsreohold =======================================

reo_hold_vol_extra_filters = None
if reo_hold_vol_extra_filters is not None:
    reo_hold_vol_extra_filters.append(date_filter)
    reo_hold_vol_final_filters = " AND ".join(
        reo_hold_vol_extra_filters
    )
else:
    reo_hold_vol_final_filters = date_filter

reo_hold_vol_selected_filters = [

    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "TimeToSell",
    "NetRecoveryAmount",
    "BusinessDate"
]

reo_hold_vol = load_landed_delta_table(
    spark,
    catalog_landed, 
    "cepalplatform_smevbusinessplandatavolcbsreohold",
    reo_hold_vol_final_filters,
    reo_hold_vol_selected_filters,
    )

# ==================================== cepalplatform_smevbusinessplandatavolcbsreosale =======================================

reo_sale_vol_extra_filters = None
if reo_sale_vol_extra_filters is not None:
    reo_sale_vol_extra_filters.append(date_filter)
    reo_sale_vol_final_filters = " AND ".join(
        reo_sale_vol_extra_filters
    )
else:
    reo_sale_vol_final_filters = date_filter

reo_sale_vol_selected_filters = [

    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "TimeToSell",
    "NetRecoveryAmount",
    "BusinessDate"
]

reo_sale_vol = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandatavolcbsreosale",
    reo_sale_vol_final_filters,
    reo_sale_vol_selected_filters,
    )

# ================== nonre_vol table schema ==================

target_nonre_vol = nonre_vol.select(
    nonre_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    nonre_vol.DebtorCDI.cast("String"),
    nonre_vol.AgreementNumber.cast("String"),
    nonre_vol.BusinessPlan_Status.cast("String"),
    nonre_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    nonre_vol.BusinessPlan_SolutionType.cast("String"),
    nonre_vol.BusinessPlan_Comments.cast("String"),
    nonre_vol.BusinessPlanStrategy_Status.cast("String"),
    nonre_vol.Confidence.cast("String"),
    nonre_vol.ProposedBy.cast("String"),
    nonre_vol.BusinessPlanStrategy_StrategyType.cast("String"),
    nonre_vol.BusinessPlanStrategy_Comments.cast("String"),
    nonre_vol.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    nonre_vol.TimeToRecovery.cast("Int").alias("TimeToRecovery"),
    nonre_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    nonre_vol.PortfolioID.cast("Int"),
    nonre_vol.BusinessDate.cast("Date"),
    lit("Business_Plan_Vol_CBS_RE_3rdParty").cast("String").alias("tbl"),
)

# ================== re_3rdparty_vol table schema ==================

target_re_3rdparty_vol = re_3rdparty_vol.select(
    re_3rdparty_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    re_3rdparty_vol.DebtorCDI.cast("String"),
    re_3rdparty_vol.AgreementNumber.cast("String"),
    re_3rdparty_vol.BusinessPlan_Status.cast("String"),
    re_3rdparty_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    re_3rdparty_vol.BusinessPlan_SolutionType.cast("String"),
    re_3rdparty_vol.BusinessPlan_Comments.cast("String"),
    re_3rdparty_vol.BusinessPlanStrategy_Status.cast("String"),
    re_3rdparty_vol.Confidence.cast("String"),
    re_3rdparty_vol.ProposedBy.cast("String"),
    re_3rdparty_vol.BusinessPlanStrategy_StrategyType.cast("String"),
    re_3rdparty_vol.BusinessPlanStrategy_Comments.cast("String"),
    re_3rdparty_vol.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    re_3rdparty_vol.TimeToSell.cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    re_3rdparty_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    re_3rdparty_vol.PortfolioID.cast("Int"),
    re_3rdparty_vol.BusinessDate.cast("Date"),
    lit("Business_Plan_Vol_CBS_RE_3rdParty").cast("String").alias("tbl"),
)

# ================== reo_hold_vol table schema ==================

target_reo_hold_vol = reo_hold_vol.select(
    reo_hold_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    reo_hold_vol.DebtorCDI.cast("String"),
    reo_hold_vol.AgreementNumber.cast("String"),
    reo_hold_vol.BusinessPlan_Status.cast("String"),
    reo_hold_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    reo_hold_vol.BusinessPlan_SolutionType.cast("String"),
    reo_hold_vol.BusinessPlan_Comments.cast("String"),
    reo_hold_vol.BusinessPlanStrategy_Status.cast("String"),
    reo_hold_vol.Confidence.cast("String"),
    reo_hold_vol.ProposedBy.cast("String"),
    reo_hold_vol.BusinessPlanStrategy_StrategyType.cast("String"),
    reo_hold_vol.BusinessPlanStrategy_Comments.cast("String"),
    reo_hold_vol.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    reo_hold_vol.TimeToSell.cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    reo_hold_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    reo_hold_vol.PortfolioID.cast("Int"),
    reo_hold_vol.BusinessDate.cast("Date"),
    lit("Business_Plan_Vol_CBS_REO_Hold").cast("String").alias("tbl"),
)

# ================== reo_sale_vol table schema ===================================

target_reo_sale_vol = reo_sale_vol.select(
    reo_sale_vol.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    reo_sale_vol.DebtorCDI.cast("String"),
    reo_sale_vol.AgreementNumber.cast("String"),
    reo_sale_vol.BusinessPlan_Status.cast("String"),
    reo_sale_vol.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    reo_sale_vol.BusinessPlan_SolutionType.cast("String"),
    reo_sale_vol.BusinessPlan_Comments.cast("String"),
    reo_sale_vol.BusinessPlanStrategy_Status.cast("String"),
    reo_sale_vol.Confidence.cast("String"),
    reo_sale_vol.ProposedBy.cast("String"),
    reo_sale_vol.BusinessPlanStrategy_StrategyType.cast("String"),
    reo_sale_vol.BusinessPlanStrategy_Comments.cast("String"),
    reo_sale_vol.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    reo_sale_vol.TimeToSell.cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    reo_sale_vol.NetRecoveryAmount.cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    reo_sale_vol.PortfolioID.cast("Int"),
    reo_sale_vol.BusinessDate.cast("Date"),
    lit("Business_Plan_Vol_CBS_REO_Sale").cast("String").alias("tbl"),
)


In [0]:
# dev-Landed:DataMode:cepalplatform_SmeVBusinessPlanDataIndivBankruptcy | cepalplatform_smevbusinessplandataindivbankruptcy
# dev-Landed:DataMode:cepalplatform_SmeVBusinessPlanDataStandStill | cepalplatform_smevbusinessplandatastandstill
# dev-Landed:DataMode:cepalplatform_SmeVBusinessPlanDataLbsPp | cepalplatform_smevbusinessplandatalbspp

In [0]:
run_date = '2026-02-18' 
date_filter = f"businessdate = {run_date!r}"

# ============================== cepalplatform_smevbusinessplandataindivbankruptcy ====================================

indiv_bankruptcy_extra_filters = None
if indiv_bankruptcy_extra_filters is not None:
    indiv_bankruptcy_extra_filters.append(date_filter)
    indiv_bankruptcy_final_filters = " AND ".join(
        indiv_bankruptcy_extra_filters
    )
else:
    indiv_bankruptcy_final_filters = date_filter

indiv_bankruptcy_selected_filters = [
    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_CreationDate",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "NumberOfInstallments",
    "Installment",
    "PrepaymentMonth",
    "PrepaymentAmount",
    "PeriodicityID",
    "PortfolioID",
    "BusinessDate",
    ]

indiv_bankruptcy = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandataindivbankruptcy",
    indiv_bankruptcy_final_filters,
    indiv_bankruptcy_selected_filters,
    )

indiv_bankruptcy = indiv_bankruptcy.withColumn(
    "PeriodicityName",
    when(col("PeriodicityID") == 1, "Μηνιαία")
    .when(col("PeriodicityID") == 2, "Διμηνιαία")
    .when(col("PeriodicityID") == 3, "Τρίμηνη")
    .when(col("PeriodicityID") == 4, "Εξάμηνη")
    .when(col("PeriodicityID") == 5, "Ετήσια")
    .otherwise(None)
)

# ============================== cepalplatform_smevbusinessplandatastandstill ====================================

stand_still_extra_filters = None
if stand_still_extra_filters is not None:
    stand_still_extra_filters.append(date_filter)
    stand_still_final_filters = " AND ".join(
        stand_still_extra_filters
    )
else:
    stand_still_final_filters = date_filter

stand_still_selected_filters = [
    "BusinessPlan_ID",
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_CreationDate",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "PortfolioID",
    "BusinessDate",
    ]

stand_still = load_landed_delta_table(
    spark,
    catalog_landed,  
    "cepalplatform_smevbusinessplandatastandstill",
    stand_still_final_filters,
    stand_still_selected_filters,
    )

# ================== indiv_bankruptcy table schema ===================================

target_indiv_bankruptcy = indiv_bankruptcy.select(
    indiv_bankruptcy.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    indiv_bankruptcy.DebtorCDI.cast("String"),
    indiv_bankruptcy.AgreementNumber.cast("String"),
    indiv_bankruptcy.BusinessPlan_Status.cast("String"),
    indiv_bankruptcy.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    indiv_bankruptcy.BusinessPlan_SolutionType.cast("String"),
    indiv_bankruptcy.BusinessPlan_Comments.cast("String"),
    indiv_bankruptcy.BusinessPlanStrategy_Status.cast("String"),
    indiv_bankruptcy.Confidence.cast("String"),
    indiv_bankruptcy.ProposedBy.cast("String"),
    indiv_bankruptcy.BusinessPlanStrategy_StrategyType.cast("String"),
    indiv_bankruptcy.BusinessPlanStrategy_Comments.cast("String"),
    indiv_bankruptcy.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    indiv_bankruptcy.NumberOfInstallments.cast("Int").alias("NumberOfInstallments"),
    indiv_bankruptcy.PrepaymentMonth.cast("Date").alias("PrepaymentMonth"),
    indiv_bankruptcy.PrepaymentAmount.cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    indiv_bankruptcy.PeriodicityName.cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    lit(None).cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    indiv_bankruptcy.PortfolioID.cast("Int"),
    indiv_bankruptcy.BusinessDate.cast("Date"),
    lit("Business_Plan_Indv_Bankcruptcy").cast("String").alias("tbl"),
)

# ================== stand_still table schema ===================================

target_stand_still = stand_still.select(
    stand_still.BusinessPlan_ID.cast("Int").alias("BusinessPlanID"),
    stand_still.DebtorCDI.cast("String"),
    stand_still.AgreementNumber.cast("String"),
    stand_still.BusinessPlan_Status.cast("String"),
    stand_still.BusinessPlanStrategy_Code.cast("Int").alias("BP_Strategy_Code"),
    stand_still.BusinessPlan_SolutionType.cast("String"),
    stand_still.BusinessPlan_Comments.cast("String"),
    stand_still.BusinessPlanStrategy_Status.cast("String"),
    stand_still.Confidence.cast("String"),
    stand_still.ProposedBy.cast("String"),
    stand_still.BusinessPlanStrategy_StrategyType.cast("String"),
    stand_still.BusinessPlanStrategy_Comments.cast("String"),
    stand_still.LastModifiedDate.cast("Date"),
    lit(None).cast("Date").alias("CollectionPlan_Month"),
    lit(None).cast("Decimal(19, 2)").alias("CollectionPlan_Amount"),
    lit(None).cast("String").alias("BusinessPlanStrategy_SolutionType"),
    lit(None).cast("Int").alias("NumberOfInstallments"),
    lit(None).cast("Date").alias("PrepaymentMonth"),
    lit(None).cast("Decimal(19, 2)").alias("PrepaymentAmount"),
    lit(None).cast("Decimal(10, 4)").alias("Interest"),
    lit(None).cast("String").alias("PeriodicityID"),
    lit(None).cast("String").alias("InterestPaymentPeriodicityID"),
    lit(None).cast("Decimal(19, 2)").alias("BalloonAmount"),
    lit(None).cast("Int").alias("TimeToAuction"),
    lit(None).cast("Decimal(19,2)").alias("AuctionPrice"),
    lit(None).cast("Int").alias("TimeToSell"),
    lit(None).cast("Int").alias("TimeToRecovery"),
    lit(None).cast("Decimal(19, 2)").alias("NetRecoveryAmount"),
    lit(None).cast("String").alias("ImplementationStatus"),
    lit(None).cast("Date").alias("FirstInstallmentDate"),
    lit(None).cast("Decimal(19, 2)").alias("StartingCapitalAmount"),
    stand_still.PortfolioID.cast("Int"),
    stand_still.BusinessDate.cast("Date"),
    lit("Business_Plan_Stand_Still").cast("String").alias("tbl"),
)

final = (
    target_business_plan_corp_bankruptcy
    .union(target_nonre_forc)
    .union(target_re_3rdparty_forc)
    .union(target_reo_hold_forc)
    .union(target_reo_sale_forc)
    .union(target_nonre_vol)
    .union(target_re_3rdparty_vol)
    .union(target_reo_hold_vol)
    .union(target_reo_sale_vol)
    .union(target_indiv_bankruptcy)
    .union(target_stand_still)
    )